# ML Notebook - Agricultural Intelligence

This notebook implements the **ML layer** of the Agri Intelligence pipeline:

```
Kafka (live APIs) → HDFS batches → Feature table → ML models
```

**Models**
| Model | Task | Algorithm |
|-------|------|-----------|
| `AnomalyDetector` | Flag extreme weather / price events | Isolation Forest |
| `CropRiskModel` | LOW / MEDIUM / HIGH risk | Random Forest Classifier |
| `YieldForecaster` | Yield (t/ha) | Random Forest Regressor |
| `RelationshipModule` | Correlations & explainability | — |

Run cells top-to-bottom. Works **offline** with synthetic data if HDFS is empty.


## 0. Setup & imports

In [ ]:
# Core deps (already in project requirements.txt)
import os
import sys
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import joblib

from sklearn.ensemble import IsolationForest, RandomForestClassifier, RandomForestRegressor
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    classification_report, accuracy_score,
    mean_absolute_error, mean_squared_error, r2_score,
)

from datetime import date, timedelta
from typing import List, Optional, Dict, Any, Tuple

# Where to save trained models
SAVED_DIR = "models_saved"
os.makedirs(SAVED_DIR, exist_ok=True)
print("Ready. Models will be saved under:", SAVED_DIR)


Ready. Models will be saved under: models_saved


## 1. Synthetic feature table (classroom / offline fallback)

Mirrors the schema of `analytics.hdfs_data_loader.get_merged_agri_dataset()`  
so the same models work on real HDFS data later.


In [ ]:
REGIONS = ["Tigray", "Amhara", "Oromia", "SNNPR", "Sidama"]
CROPS = ["Teff", "Maize", "Wheat", "Coffee", "Sorghum"]

def generate_synthetic_merged_dataset(n_days: int = 40, seed: int = 42) -> pd.DataFrame:
    rng = np.random.default_rng(seed)
    rows = []
    start = date.today() - timedelta(days=n_days)

    for d in range(n_days):
        day = start + timedelta(days=d)
        for region in REGIONS:
            base_temp = rng.uniform(16, 30)
            rain = max(0.0, rng.normal(4.0, 6.0))
            humidity = rng.uniform(30, 90)
            soil = rng.uniform(12, 55)

            for crop in CROPS:
                pest = float(np.clip(rng.beta(2, 5), 0.05, 0.95))
                veg = float(np.clip(rng.beta(5, 2), 0.15, 0.95))
                veg = float(np.clip(veg + 0.01 * rain - 0.005 * (base_temp - 22), 0.1, 0.95))
                pest = float(np.clip(pest + 0.008 * (base_temp - 20), 0.05, 0.95))

                price = {
                    "Teff": 5200, "Maize": 3800, "Wheat": 4200,
                    "Coffee": 8500, "Sorghum": 3400,
                }[crop] + rng.normal(0, 120)
                vol = abs(rng.normal(0, 80))

                area = rng.uniform(200, 1200)
                production = (1.5 + 0.07 * rain + 2.0 * veg - 1.2 * pest) * area
                production = max(50.0, production + rng.normal(0, 40))

                rows.append({
                    "region": region,
                    "crop": crop,
                    "date": day,
                    "avg_temp": round(base_temp + rng.normal(0, 0.8), 2),
                    "max_temp": round(base_temp + rng.uniform(2, 6), 2),
                    "min_temp": round(base_temp - rng.uniform(2, 5), 2),
                    "avg_rainfall": round(rain, 2),
                    "total_rainfall": round(rain * rng.uniform(0.8, 1.4), 2),
                    "avg_humidity": round(humidity, 2),
                    "avg_soil_moisture": round(soil, 2),
                    "min_soil_moisture": round(max(5.0, soil - rng.uniform(3, 10)), 2),
                    "avg_pest_threat": round(pest, 3),
                    "avg_veg_health": round(veg, 3),
                    "avg_price": round(price, 2),
                    "max_price": round(price + abs(rng.normal(0, 50)), 2),
                    "min_price": round(price - abs(rng.normal(0, 50)), 2),
                    "price_volatility": round(vol, 2),
                    "total_volume": round(abs(rng.normal(80, 40)), 2),
                    "production_tons": round(production, 2),
                    "area_hectares": round(area, 2),
                })

    df = pd.DataFrame(rows)
    print(f"Generated {len(df)} rows ({n_days} days × {len(REGIONS)} regions × {len(CROPS)} crops)")
    return df

df = generate_synthetic_merged_dataset()
df.head(8)


Generated 1000 rows (40 days × 5 regions × 5 crops)


,region,crop,date,avg_temp,max_temp,min_temp,avg_rainfall,total_rainfall,avg_humidity,avg_soil_moisture,min_soil_moisture,avg_pest_threat,avg_veg_health,avg_price,max_price,min_price,price_volatility,total_volume,production_tons,area_hectares
0,Tigray,Teff,2026-07-03,27.54,31.36,22.56,0.00,0.00,81.52,41.99,32.19,0.339,0.772,5256.10,5263.83,5234.68,68.74,65.91,1951.39,754.58
1,Tigray,Maize,2026-07-03,27.43,32.17,22.73,0.00,0.00,81.52,41.99,33.16,0.359,0.560,3786.33,3797.93,3780.49,67.21,88.75,960.77,426.91
2,Tigray,Wheat,2026-07-03,26.14,30.72,23.14,0.00,0.00,81.52,41.99,34.54,0.382,0.905,4143.56,4172.87,4107.99,51.11,111.73,1028.81,339.80
3,Tigray,Coffee,2026-07-03,27.34,30.46,22.39,0.00,0.00,81.52,41.99,38.83,0.211,0.742,8517.09,8536.18,8457.30,55.24,99.48,2073.14,757.03
4,Tigray,Sorghum,2026-07-03,27.16,31.63,24.04,0.00,0.00,81.52,41.99,33.54,0.218,0.817,3197.52,3234.40,3150.84,115.77,71.78,873.71,318.01
5,Amhara,Teff,2026-07-03,17.53,21.33,14.36,1.97,1.74,57.35,20.70,16.98,0.375,0.751,5392.21,5451.65,5360.23,19.15,42.94,772.49,284.44
6,Amhara,Maize,2026-07-03,15.64,20.42,14.35,1.97,2.19,57.35,20.70,14.63,0.143,0.549,4039.61,4046.65,3986.31,233.11,86.28,2877.73,1137.83
7,Amhara,Wheat,2026-07-03,19.05,22.67,12.92,1.97,1.95,57.35,20.70,11.03,0.415,0.755,4140.08,4159.33,4132.25,94.80,78.37,600.14,237.41


In [ ]:
print("Shape:", df.shape)
print("\nColumns:", list(df.columns))
print("\nRegions:", sorted(df["region"].unique()))
print("Crops:", sorted(df["crop"].unique()))
df.describe().round(2)


Shape: (1000, 20)

Columns: ['region', 'crop', 'date', 'avg_temp', 'max_temp', 'min_temp', 'avg_rainfall', 'total_rainfall', 'avg_humidity', 'avg_soil_moisture', 'min_soil_moisture', 'avg_pest_threat', 'avg_veg_health', 'avg_price', 'max_price', 'min_price', 'price_volatility', 'total_volume', 'production_tons', 'area_hectares']

Regions: ['Amhara', 'Oromia', 'SNNPR', 'Sidama', 'Tigray']
Crops: ['Coffee', 'Maize', 'Sorghum', 'Teff', 'Wheat']


,avg_temp,max_temp,min_temp,avg_rainfall,total_rainfall,avg_humidity,avg_soil_moisture,min_soil_moisture,avg_pest_threat,avg_veg_health,avg_price,max_price,min_price,price_volatility,total_volume,production_tons,area_hectares
count,1000.00,1000.00,1000.00,1000.00,1000.00,1000.00,1000.00,1000.00,1000.00,1000.00,1000.00,1000.00,1000.00,1000.00,1000.00,1000.00,1000.00
mean,22.84,26.82,19.37,4.60,5.06,61.84,32.51,25.92,0.31,0.75,5021.67,5060.31,4982.40,64.30,80.24,2044.93,691.50
std,3.88,3.92,3.87,4.56,5.13,16.29,12.07,12.25,0.16,0.15,1842.63,1843.25,1843.80,47.91,37.40,961.10,295.96
min,14.79,18.53,11.40,0.00,0.00,30.38,12.12,5.00,0.05,0.24,3099.59,3118.25,3054.24,0.14,0.11,394.46,200.52
25%,19.38,23.66,16.12,0.00,0.00,47.82,22.38,16.15,0.19,0.65,3727.03,3764.39,3674.97,26.26,53.55,1259.11,426.05
50%,23.24,26.98,19.70,4.00,4.09,62.02,31.70,25.00,0.28,0.77,4204.88,4242.70,4163.10,55.49,77.94,1980.66,680.90
75%,25.93,29.78,22.56,7.10,8.00,75.36,42.79,35.81,0.42,0.88,5298.44,5341.68,5264.00,93.01,105.26,2757.35,955.57
max,30.92,35.76,27.42,18.89,23.18,89.97,54.80,51.46,0.86,0.95,8789.35,8818.90,8787.62,248.25,196.20,4907.40,1198.87


## 2. Anomaly Detector (Isolation Forest)

**Why Isolation Forest?** Unsupervised, works without labels, `contamination` is an explicit assumption you can defend in the viva (~5 % extreme events).


In [ ]:
DEFAULT_FEATURE_COLUMNS = [
    "avg_temp", "max_temp", "min_temp",
    "avg_rainfall", "total_rainfall",
    "avg_humidity", "avg_soil_moisture", "min_soil_moisture",
    "price_volatility", "avg_price",
]


class AnomalyDetector:
    def __init__(
        self,
        contamination: float = 0.05,
        n_estimators: int = 200,
        random_state: int = 42,
        feature_columns: Optional[List[str]] = None,
    ):
        if not 0.0 < contamination < 0.5:
            raise ValueError("contamination should be between 0 and 0.5")
        self.contamination = contamination
        self.n_estimators = n_estimators
        self.random_state = random_state
        self.feature_columns = feature_columns or list(DEFAULT_FEATURE_COLUMNS)
        self.scaler = StandardScaler()
        self.model = IsolationForest(
            n_estimators=self.n_estimators,
            contamination=self.contamination,
            random_state=self.random_state,
            n_jobs=-1,
        )
        self._is_fitted = False
        self._used_features: List[str] = []

    def _select_features(self, df: pd.DataFrame) -> pd.DataFrame:
        available = [c for c in self.feature_columns if c in df.columns]
        if not available:
            raise ValueError(f"None of {self.feature_columns} found. Available: {list(df.columns)}")
        self._used_features = available
        X = df[available].copy().fillna(df[available].median(numeric_only=True))
        return X

    def fit(self, df: pd.DataFrame) -> "AnomalyDetector":
        if df is None or df.empty:
            raise ValueError("Empty DataFrame")
        X = self._select_features(df)
        self.model.fit(self.scaler.fit_transform(X))
        self._is_fitted = True
        return self

    def predict(self, df: pd.DataFrame) -> np.ndarray:
        self._check_fitted()
        X = self._select_features(df)
        return self.model.predict(self.scaler.transform(X))

    def decision_function(self, df: pd.DataFrame) -> np.ndarray:
        self._check_fitted()
        X = self._select_features(df)
        return self.model.decision_function(self.scaler.transform(X))

    def score_and_rank(self, df: pd.DataFrame) -> pd.DataFrame:
        self._check_fitted()
        out = df.copy()
        out["anomaly_label"] = self.predict(df)
        out["anomaly_score"] = self.decision_function(df)
        return out.sort_values("anomaly_score", ascending=True).reset_index(drop=True)

    def explain_top_anomalies(self, df: pd.DataFrame, top_n: int = 5) -> List[Dict[str, Any]]:
        ranked = self.score_and_rank(df).head(top_n)
        summaries = []
        for _, row in ranked.iterrows():
            summary = {
                "region": row.get("region"),
                "crop": row.get("crop"),
                "date": str(row.get("date")),
                "anomaly_score": float(row["anomaly_score"]),
                "key_signals": {c: float(row[c]) for c in self._used_features
                                if c in row and pd.notna(row[c])},
            }
            summaries.append(summary)
        return summaries

    def save(self, path: str) -> None:
        self._check_fitted()
        os.makedirs(os.path.dirname(path) or ".", exist_ok=True)
        joblib.dump({
            "model": self.model, "scaler": self.scaler,
            "feature_columns": self.feature_columns,
            "used_features": self._used_features,
            "contamination": self.contamination,
            "n_estimators": self.n_estimators,
            "random_state": self.random_state,
        }, path)
        print(f"[ANOMALY] Saved → {path}")

    @classmethod
    def load(cls, path: str) -> "AnomalyDetector":
        p = joblib.load(path)
        obj = cls(p["contamination"], p["n_estimators"], p["random_state"], p["feature_columns"])
        obj.model, obj.scaler = p["model"], p["scaler"]
        obj._used_features, obj._is_fitted = p["used_features"], True
        print(f"[ANOMALY] Loaded ← {path}")
        return obj

    def _check_fitted(self):
        if not self._is_fitted:
            raise RuntimeError("Call fit() first.")


In [ ]:
anomaly = AnomalyDetector(contamination=0.05, n_estimators=200)
anomaly.fit(df)

ranked = anomaly.score_and_rank(df)
n_anom = int((ranked["anomaly_label"] == -1).sum())
print(f"Flagged {n_anom} / {len(df)} rows as anomalous ({100*n_anom/len(df):.1f}%)")

print("\nTop 5 most anomalous:")
display(ranked[["region", "crop", "date", "avg_temp", "total_rainfall",
                "avg_pest_threat", "anomaly_score", "anomaly_label"]].head(5))

anomaly.save(os.path.join(SAVED_DIR, "anomaly_detector.joblib"))


Flagged 50 / 1000 rows as anomalous (5.0%)

Top 5 most anomalous:


,region,crop,date,avg_temp,total_rainfall,avg_pest_threat,anomaly_score,anomaly_label
0,Tigray,Coffee,2026-07-13,30.72,15.60,0.270,-0.064059,-1
1,Amhara,Teff,2026-07-28,19.08,21.97,0.694,-0.053921,-1
2,Oromia,Sorghum,2026-07-30,29.94,21.35,0.397,-0.052616,-1
3,Oromia,Coffee,2026-07-30,28.21,18.80,0.571,-0.051786,-1
4,Oromia,Teff,2026-07-30,29.65,20.17,0.484,-0.050835,-1


[ANOMALY] Saved → models_saved/anomaly_detector.joblib


## 3. Crop Risk Model (Random Forest Classifier)

**Silver-standard label** (defendable in viva):

```
risk_score = 0.45*pest + 0.30*(1-veg) + 0.15*heat_stress + 0.10*moisture_stress
LOW < 0.35 ≤ MEDIUM < 0.65 ≤ HIGH
```


In [ ]:
RISK_FEATURE_COLUMNS = [
    "avg_temp", "max_temp", "min_temp",
    "avg_rainfall", "total_rainfall",
    "avg_humidity", "avg_soil_moisture", "min_soil_moisture",
    "avg_pest_threat", "avg_veg_health",
    "avg_price", "price_volatility",
]
RISK_LABELS = ["LOW", "MEDIUM", "HIGH"]


class CropRiskModel:
    def __init__(
        self,
        n_estimators: int = 150,
        max_depth: Optional[int] = 8,
        random_state: int = 42,
        feature_columns: Optional[List[str]] = None,
    ):
        self.n_estimators = n_estimators
        self.max_depth = max_depth
        self.random_state = random_state
        self.feature_columns = feature_columns or list(RISK_FEATURE_COLUMNS)
        self.model = RandomForestClassifier(
            n_estimators=self.n_estimators, max_depth=self.max_depth,
            random_state=self.random_state, class_weight="balanced", n_jobs=-1,
        )
        self.label_encoder = LabelEncoder()
        self.label_encoder.fit(RISK_LABELS)
        self._is_fitted = False
        self._used_features: List[str] = []
        self._categorical_columns = ["region", "crop"]
        self._feature_names_after_encoding: List[str] = []

    @staticmethod
    def derive_risk_labels(df: pd.DataFrame) -> pd.Series:
        pest = df.get("avg_pest_threat", pd.Series(0.3, index=df.index)).fillna(0.3)
        veg = df.get("avg_veg_health", pd.Series(0.5, index=df.index)).fillna(0.5)
        max_temp = df.get("max_temp", pd.Series(25.0, index=df.index)).fillna(25.0)
        heat = ((max_temp - 28.0).clip(lower=0) / 15.0).clip(0, 1)
        soil = df.get("avg_soil_moisture", pd.Series(35.0, index=df.index)).fillna(35.0)
        moisture_stress = (1.0 - (soil / 60.0)).clip(0, 1)
        score = 0.45 * pest + 0.30 * (1.0 - veg) + 0.15 * heat + 0.10 * moisture_stress
        return pd.cut(score, bins=[-0.01, 0.35, 0.65, 1.01], labels=RISK_LABELS).astype(str)

    def _build_feature_matrix(self, df: pd.DataFrame, fit_encoder: bool = False):
        numeric_available = [c for c in self.feature_columns if c in df.columns]
        if not numeric_available:
            raise ValueError(f"No usable features from {self.feature_columns}")
        X_num = df[numeric_available].copy().fillna(df[numeric_available].median(numeric_only=True))
        cat_parts = []
        for col in self._categorical_columns:
            if col in df.columns:
                cat_parts.append(pd.get_dummies(df[col].astype(str), prefix=col))
        X = pd.concat([X_num] + cat_parts, axis=1) if cat_parts else X_num
        if not fit_encoder and self._feature_names_after_encoding:
            for col in self._feature_names_after_encoding:
                if col not in X.columns:
                    X[col] = 0
            X = X[self._feature_names_after_encoding]
        if fit_encoder:
            self._feature_names_after_encoding = list(X.columns)
            self._used_features = numeric_available
        return X, list(X.columns)

    def fit(self, df: pd.DataFrame, test_size: float = 0.2) -> Dict[str, Any]:
        if df is None or df.empty:
            raise ValueError("Empty DataFrame")
        y_str = self.derive_risk_labels(df)
        y = self.label_encoder.transform(y_str)
        X, _ = self._build_feature_matrix(df, fit_encoder=True)
        X_train, X_test, y_train, y_test = train_test_split(
            X, y, test_size=test_size, random_state=self.random_state, stratify=y
        )
        self.model.fit(X_train, y_train)
        self._is_fitted = True
        y_pred = self.model.predict(X_test)
        metrics = {
            "accuracy": float(accuracy_score(y_test, y_pred)),
            "report": classification_report(
                y_test, y_pred,
                labels=list(range(len(self.label_encoder.classes_))),
                target_names=list(self.label_encoder.classes_),
                output_dict=True, zero_division=0,
            ),
            "n_train": int(len(X_train)), "n_test": int(len(X_test)),
            "class_distribution": {lbl: int((y_str == lbl).sum()) for lbl in RISK_LABELS},
        }
        print(f"[CROP-RISK] Hold-out accuracy = {metrics['accuracy']:.3f} "
              f"(train={metrics['n_train']}, test={metrics['n_test']})")
        return metrics

    def predict(self, df: pd.DataFrame) -> np.ndarray:
        self._check_fitted()
        X, _ = self._build_feature_matrix(df, fit_encoder=False)
        return self.label_encoder.inverse_transform(self.model.predict(X))

    def predict_proba(self, df: pd.DataFrame) -> pd.DataFrame:
        self._check_fitted()
        X, _ = self._build_feature_matrix(df, fit_encoder=False)
        return pd.DataFrame(self.model.predict_proba(X),
                            columns=self.label_encoder.classes_, index=df.index)

    def predict_with_details(self, df: pd.DataFrame) -> pd.DataFrame:
        out = df.copy()
        out["predicted_risk"] = self.predict(df)
        proba = self.predict_proba(df)
        for col in proba.columns:
            out[f"prob_{col}"] = proba[col].values
        return out

    def feature_importances(self) -> pd.DataFrame:
        self._check_fitted()
        return (pd.DataFrame({
            "feature": self._feature_names_after_encoding,
            "importance": self.model.feature_importances_,
        }).sort_values("importance", ascending=False).reset_index(drop=True))

    def save(self, path: str) -> None:
        self._check_fitted()
        os.makedirs(os.path.dirname(path) or ".", exist_ok=True)
        joblib.dump({
            "model": self.model, "label_encoder": self.label_encoder,
            "feature_columns": self.feature_columns,
            "used_features": self._used_features,
            "feature_names_after_encoding": self._feature_names_after_encoding,
            "n_estimators": self.n_estimators, "max_depth": self.max_depth,
            "random_state": self.random_state,
        }, path)
        print(f"[CROP-RISK] Saved → {path}")

    @classmethod
    def load(cls, path: str) -> "CropRiskModel":
        p = joblib.load(path)
        obj = cls(p["n_estimators"], p["max_depth"], p["random_state"], p["feature_columns"])
        obj.model = p["model"]
        obj.label_encoder = p["label_encoder"]
        obj._used_features = p["used_features"]
        obj._feature_names_after_encoding = p["feature_names_after_encoding"]
        obj._is_fitted = True
        print(f"[CROP-RISK] Loaded ← {path}")
        return obj

    def _check_fitted(self):
        if not self._is_fitted:
            raise RuntimeError("Call fit() first.")


In [ ]:
# =====================================================================
# CELL 3: Crop Risk Model (Random Forest Classifier) - FIXED
# =====================================================================

RISK_FEATURE_COLUMNS = [
    "avg_temp", "max_temp", "min_temp",
    "avg_rainfall", "total_rainfall",
    "avg_humidity", "avg_soil_moisture", "min_soil_moisture",
    "avg_pest_threat", "avg_veg_health",
    "avg_price", "price_volatility",
]
RISK_LABELS = ["LOW", "MEDIUM", "HIGH"]


class CropRiskModel:
    def __init__(
        self,
        n_estimators: int = 150,
        max_depth: Optional[int] = 8,
        random_state: int = 42,
        feature_columns: Optional[List[str]] = None,
    ):
        self.n_estimators = n_estimators
        self.max_depth = max_depth
        self.random_state = random_state
        self.feature_columns = feature_columns or list(RISK_FEATURE_COLUMNS)
        self.model = RandomForestClassifier(
            n_estimators=self.n_estimators, max_depth=self.max_depth,
            random_state=self.random_state, class_weight="balanced", n_jobs=-1,
        )
        self.label_encoder = LabelEncoder()
        self.label_encoder.fit(RISK_LABELS)
        self._is_fitted = False
        self._used_features: List[str] = []
        self._categorical_columns = ["region", "crop"]
        self._feature_names_after_encoding: List[str] = []

    @staticmethod
    def derive_risk_labels(df: pd.DataFrame) -> pd.Series:
        pest = df.get("avg_pest_threat", pd.Series(0.3, index=df.index)).fillna(0.3)
        veg = df.get("avg_veg_health", pd.Series(0.5, index=df.index)).fillna(0.5)
        max_temp = df.get("max_temp", pd.Series(25.0, index=df.index)).fillna(25.0)
        heat = ((max_temp - 28.0).clip(lower=0) / 15.0).clip(0, 1)
        soil = df.get("avg_soil_moisture", pd.Series(35.0, index=df.index)).fillna(35.0)
        moisture_stress = (1.0 - (soil / 60.0)).clip(0, 1)
        score = 0.45 * pest + 0.30 * (1.0 - veg) + 0.15 * heat + 0.10 * moisture_stress
        return pd.cut(score, bins=[-0.01, 0.35, 0.65, 1.01], labels=RISK_LABELS).astype(str)

    def _build_feature_matrix(self, df: pd.DataFrame, fit_encoder: bool = False):
        numeric_available = [c for c in self.feature_columns if c in df.columns]
        if not numeric_available:
            raise ValueError(f"No usable features from {self.feature_columns}")
        X_num = df[numeric_available].copy().fillna(df[numeric_available].median(numeric_only=True))
        cat_parts = []
        for col in self._categorical_columns:
            if col in df.columns:
                cat_parts.append(pd.get_dummies(df[col].astype(str), prefix=col))
        X = pd.concat([X_num] + cat_parts, axis=1) if cat_parts else X_num
        if not fit_encoder and self._feature_names_after_encoding:
            for col in self._feature_names_after_encoding:
                if col not in X.columns:
                    X[col] = 0
            X = X[self._feature_names_after_encoding]
        if fit_encoder:
            self._feature_names_after_encoding = list(X.columns)
            self._used_features = numeric_available
        return X, list(X.columns)

    def fit(self, df: pd.DataFrame, test_size: float = 0.2) -> Dict[str, Any]:
        if df is None or df.empty:
            raise ValueError("Empty DataFrame")
        y_str = self.derive_risk_labels(df)
        y = self.label_encoder.transform(y_str)
        X, _ = self._build_feature_matrix(df, fit_encoder=True)
        X_train, X_test, y_train, y_test = train_test_split(
            X, y, test_size=test_size, random_state=self.random_state, stratify=y
        )
        self.model.fit(X_train, y_train)
        self._is_fitted = True
        y_pred = self.model.predict(X_test)
        metrics = {
            "accuracy": float(accuracy_score(y_test, y_pred)),
            "report": classification_report(
                y_test, y_pred,
                labels=list(range(len(self.label_encoder.classes_))),
                target_names=list(self.label_encoder.classes_),
                output_dict=True, zero_division=0,
            ),
            "n_train": int(len(X_train)), "n_test": int(len(X_test)),
            "class_distribution": {lbl: int((y_str == lbl).sum()) for lbl in RISK_LABELS},
        }
        print(f"[CROP-RISK] Hold-out accuracy = {metrics['accuracy']:.3f} "
              f"(train={metrics['n_train']}, test={metrics['n_test']})")
        return metrics

    def predict(self, df: pd.DataFrame) -> np.ndarray:
        self._check_fitted()
        X, _ = self._build_feature_matrix(df, fit_encoder=False)
        return self.label_encoder.inverse_transform(self.model.predict(X))

    def predict_proba(self, df: pd.DataFrame) -> pd.DataFrame:
        """Safely maps probabilities for all classes, handling cases where a class was absent in training."""
        self._check_fitted()
        X, _ = self._build_feature_matrix(df, fit_encoder=False)
        raw_probs = self.model.predict_proba(X)

        # Map probabilities using the exact classes present in the fitted model
        fitted_class_names = self.label_encoder.inverse_transform(self.model.classes_)

        # Create a DataFrame initialized with 0.0 for ALL possible target classes
        proba_df = pd.DataFrame(0.0, index=df.index, columns=self.label_encoder.classes_)

        # Populate columns for the classes that actually exist in the model
        for idx, col_name in enumerate(fitted_class_names):
            proba_df[col_name] = raw_probs[:, idx]

        return proba_df

    def predict_with_details(self, df: pd.DataFrame) -> pd.DataFrame:
        out = df.copy()
        out["predicted_risk"] = self.predict(df)
        proba = self.predict_proba(df)
        for col in proba.columns:
            out[f"prob_{col}"] = proba[col].values
        return out

    def feature_importances(self) -> pd.DataFrame:
        self._check_fitted()
        return (pd.DataFrame({
            "feature": self._feature_names_after_encoding,
            "importance": self.model.feature_importances_,
        }).sort_values("importance", ascending=False).reset_index(drop=True))

    def save(self, path: str) -> None:
        self._check_fitted()
        os.makedirs(os.path.dirname(path) or ".", exist_ok=True)
        joblib.dump({
            "model": self.model, "label_encoder": self.label_encoder,
            "feature_columns": self.feature_columns,
            "used_features": self._used_features,
            "feature_names_after_encoding": self._feature_names_after_encoding,
            "n_estimators": self.n_estimators, "max_depth": self.max_depth,
            "random_state": self.random_state,
        }, path)
        print(f"[CROP-RISK] Saved → {path}")

    @classmethod
    def load(cls, path: str) -> "CropRiskModel":
        p = joblib.load(path)
        obj = cls(p["n_estimators"], p["max_depth"], p["random_state"], p["feature_columns"])
        obj.model = p["model"]
        obj.label_encoder = p["label_encoder"]
        obj._used_features = p["used_features"]
        obj._feature_names_after_encoding = p["feature_names_after_encoding"]
        obj._is_fitted = True
        print(f"[CROP-RISK] Loaded ← {path}")
        return obj

    def _check_fitted(self):
        if not self._is_fitted:
            raise RuntimeError("Call fit() first.")

## 4. Yield Forecaster (Random Forest Regressor)

Target = `production_tons / area_hectares` (t/ha).  
Falls back to a rainfall + vegetation proxy only if production columns are missing.


In [ ]:
YIELD_FEATURE_COLUMNS = [
    "avg_temp", "max_temp", "min_temp",
    "avg_rainfall", "total_rainfall",
    "avg_humidity", "avg_soil_moisture", "min_soil_moisture",
    "avg_pest_threat", "avg_veg_health",
]


class YieldForecaster:
    def __init__(
        self,
        n_estimators: int = 150,
        max_depth: Optional[int] = 10,
        random_state: int = 42,
        feature_columns: Optional[List[str]] = None,
    ):
        self.n_estimators = n_estimators
        self.max_depth = max_depth
        self.random_state = random_state
        self.feature_columns = feature_columns or list(YIELD_FEATURE_COLUMNS)
        self.model = RandomForestRegressor(
            n_estimators=self.n_estimators, max_depth=self.max_depth,
            random_state=self.random_state, n_jobs=-1,
        )
        self._is_fitted = False
        self._used_features: List[str] = []
        self._categorical_columns = ["region", "crop"]
        self._feature_names_after_encoding: List[str] = []
        self._target_name = "yield_t_ha"

    @staticmethod
    def build_target(df: pd.DataFrame) -> pd.Series:
        if "production_tons" in df.columns and "area_hectares" in df.columns:
            prod = df["production_tons"].astype(float)
            area = df["area_hectares"].astype(float).replace(0, np.nan)
            yield_t_ha = (prod / area).clip(lower=0.1, upper=15.0)
            if yield_t_ha.notna().sum() >= max(10, int(0.3 * len(df))):
                return yield_t_ha.fillna(yield_t_ha.median())
        rain = df.get("total_rainfall", pd.Series(5.0, index=df.index)).fillna(5.0)
        veg = df.get("avg_veg_health", pd.Series(0.5, index=df.index)).fillna(0.5)
        pest = df.get("avg_pest_threat", pd.Series(0.3, index=df.index)).fillna(0.3)
        base = 1.2 + 0.08 * rain + 2.5 * veg - 1.5 * pest
        noise = np.random.default_rng(42).normal(0, 0.25, size=len(df))
        print("[YIELD] Using rainfall + vegetation synthetic target.")
        return pd.Series((base + noise).clip(0.3, 8.0), index=df.index, name="yield_t_ha")

    def _build_feature_matrix(self, df: pd.DataFrame, fit_encoder: bool = False) -> pd.DataFrame:
        numeric_available = [c for c in self.feature_columns if c in df.columns]
        if not numeric_available:
            raise ValueError(f"No usable features from {self.feature_columns}")
        X_num = df[numeric_available].copy().fillna(df[numeric_available].median(numeric_only=True))
        cat_parts = []
        for col in self._categorical_columns:
            if col in df.columns:
                cat_parts.append(pd.get_dummies(df[col].astype(str), prefix=col))
        X = pd.concat([X_num] + cat_parts, axis=1) if cat_parts else X_num
        if not fit_encoder and self._feature_names_after_encoding:
            for col in self._feature_names_after_encoding:
                if col not in X.columns:
                    X[col] = 0
            X = X[self._feature_names_after_encoding]
        if fit_encoder:
            self._feature_names_after_encoding = list(X.columns)
            self._used_features = numeric_available
        return X

    def fit(self, df: pd.DataFrame, test_size: float = 0.2) -> Dict[str, Any]:
        if df is None or df.empty:
            raise ValueError("Empty DataFrame")
        y = self.build_target(df)
        X = self._build_feature_matrix(df, fit_encoder=True)
        X_train, X_test, y_train, y_test = train_test_split(
            X, y, test_size=test_size, random_state=self.random_state
        )
        self.model.fit(X_train, y_train)
        self._is_fitted = True
        y_pred = self.model.predict(X_test)
        metrics = {
            "mae": float(mean_absolute_error(y_test, y_pred)),
            "rmse": float(np.sqrt(mean_squared_error(y_test, y_pred))),
            "r2": float(r2_score(y_test, y_pred)),
            "n_train": int(len(X_train)), "n_test": int(len(X_test)),
            "target_mean": float(y.mean()), "target_std": float(y.std()),
        }
        print(f"[YIELD] R²={metrics['r2']:.3f}, MAE={metrics['mae']:.3f} t/ha "
              f"(train={metrics['n_train']}, test={metrics['n_test']})")
        return metrics

    def predict(self, df: pd.DataFrame) -> np.ndarray:
        self._check_fitted()
        return self.model.predict(self._build_feature_matrix(df, fit_encoder=False))

    def predict_with_details(self, df: pd.DataFrame) -> pd.DataFrame:
        out = df.copy()
        out["predicted_yield_t_ha"] = self.predict(df)
        if "production_tons" in out.columns and "area_hectares" in out.columns:
            area = out["area_hectares"].replace(0, np.nan)
            out["actual_yield_t_ha"] = (out["production_tons"] / area).clip(0.1, 15.0)
        return out

    def feature_importances(self) -> pd.DataFrame:
        self._check_fitted()
        return (pd.DataFrame({
            "feature": self._feature_names_after_encoding,
            "importance": self.model.feature_importances_,
        }).sort_values("importance", ascending=False).reset_index(drop=True))

    def save(self, path: str) -> None:
        self._check_fitted()
        os.makedirs(os.path.dirname(path) or ".", exist_ok=True)
        joblib.dump({
            "model": self.model, "feature_columns": self.feature_columns,
            "used_features": self._used_features,
            "feature_names_after_encoding": self._feature_names_after_encoding,
            "n_estimators": self.n_estimators, "max_depth": self.max_depth,
            "random_state": self.random_state, "target_name": self._target_name,
        }, path)
        print(f"[YIELD] Saved → {path}")

    @classmethod
    def load(cls, path: str) -> "YieldForecaster":
        p = joblib.load(path)
        obj = cls(p["n_estimators"], p["max_depth"], p["random_state"], p["feature_columns"])
        obj.model = p["model"]
        obj._used_features = p["used_features"]
        obj._feature_names_after_encoding = p["feature_names_after_encoding"]
        obj._target_name = p.get("target_name", "yield_t_ha")
        obj._is_fitted = True
        print(f"[YIELD] Loaded ← {path}")
        return obj

    def _check_fitted(self):
        if not self._is_fitted:
            raise RuntimeError("Call fit() first.")


In [ ]:
yield_model = YieldForecaster(n_estimators=150, max_depth=10)
yield_metrics = yield_model.fit(df)

print("\nTop yield drivers:")
display(yield_model.feature_importances().head(10))

print("\nSample predictions vs actual:")
display(yield_model.predict_with_details(df)[
    ["region", "crop", "total_rainfall", "avg_veg_health",
     "actual_yield_t_ha", "predicted_yield_t_ha"]
].head(8))

yield_model.save(os.path.join(SAVED_DIR, "yield_forecaster.joblib"))


[YIELD] R²=0.967, MAE=0.078 t/ha (train=800, test=200)

Top yield drivers:


,feature,importance
0,avg_rainfall,0.435212
1,avg_veg_health,0.375034
2,avg_pest_threat,0.119087
3,total_rainfall,0.043541
4,avg_humidity,0.004479
5,avg_temp,0.003568
6,max_temp,0.003397
7,min_soil_moisture,0.003379
8,min_temp,0.003177
9,avg_soil_moisture,0.002960



Sample predictions vs actual:


,region,crop,total_rainfall,avg_veg_health,actual_yield_t_ha,predicted_yield_t_ha
0,Tigray,Teff,0.00,0.772,2.586061,2.637581
1,Tigray,Maize,0.00,0.560,2.250521,2.250887
2,Tigray,Wheat,0.00,0.905,3.027693,2.958026
3,Tigray,Coffee,0.00,0.742,2.738518,2.753562
4,Tigray,Sorghum,0.00,0.817,2.747429,2.815399
5,Amhara,Teff,1.74,0.751,2.715828,2.688814
6,Amhara,Maize,2.19,0.549,2.529139,2.504990
7,Amhara,Wheat,1.95,0.755,2.527863,2.602345


[YIELD] Saved → models_saved/yield_forecaster.joblib


## 5. Relationship / Explainability Module


In [ ]:
NUMERIC_CANDIDATES = [
    "avg_temp", "max_temp", "min_temp",
    "avg_rainfall", "total_rainfall",
    "avg_humidity", "avg_soil_moisture", "min_soil_moisture",
    "avg_pest_threat", "avg_veg_health",
    "avg_price", "price_volatility", "total_volume",
    "production_tons", "area_hectares",
]


class RelationshipModule:
    def __init__(self, df: Optional[pd.DataFrame] = None):
        self.df = df.copy() if df is not None else None

    def set_data(self, df: pd.DataFrame) -> "RelationshipModule":
        self.df = df.copy()
        return self

    def correlation_matrix(self, columns=None, method="pearson") -> pd.DataFrame:
        self._require_data()
        cols = columns or [c for c in NUMERIC_CANDIDATES if c in self.df.columns]
        return self.df[cols].corr(method=method) if len(cols) >= 2 else pd.DataFrame()

    def top_correlations(self, target: str, top_n: int = 8, method="pearson") -> pd.DataFrame:
        self._require_data()
        if target not in self.df.columns:
            raise ValueError(f"'{target}' not in DataFrame")
        cols = [c for c in NUMERIC_CANDIDATES if c in self.df.columns and c != target]
        if not cols:
            return pd.DataFrame()
        corr = self.df[cols + [target]].corr(method=method)[target].drop(target)
        return (pd.DataFrame({"feature": corr.index, "correlation": corr.values})
                .assign(abs_corr=lambda d: d["correlation"].abs())
                .sort_values("abs_corr", ascending=False).head(top_n)
                .drop(columns=["abs_corr"]).reset_index(drop=True))

    def region_summary(self) -> pd.DataFrame:
        self._require_data()
        if "region" not in self.df.columns:
            return pd.DataFrame()
        agg = {c: "mean" for c in ["avg_temp", "total_rainfall", "avg_soil_moisture",
                                    "avg_pest_threat", "avg_veg_health", "avg_price"]
               if c in self.df.columns}
        return self.df.groupby("region", as_index=False).agg(agg).round(3) if agg else pd.DataFrame()

    def crop_summary(self) -> pd.DataFrame:
        self._require_data()
        if "crop" not in self.df.columns:
            return pd.DataFrame()
        agg = {c: "mean" for c in ["avg_pest_threat", "avg_veg_health", "avg_price",
                                    "price_volatility", "total_rainfall"]
               if c in self.df.columns}
        return self.df.groupby("crop", as_index=False).agg(agg).round(3) if agg else pd.DataFrame()

    def explain_risk_drivers(self, risk_model, top_n=10):
        return risk_model.feature_importances().head(top_n)

    def explain_yield_drivers(self, yield_model, top_n=10):
        return yield_model.feature_importances().head(top_n)

    def summarise_anomalies(self, anomaly_detector, top_n=5):
        self._require_data()
        return anomaly_detector.explain_top_anomalies(self.df, top_n=top_n)

    def narrative_report(self, risk_model=None, yield_model=None, anomaly_detector=None) -> str:
        self._require_data()
        lines = ["=== Agricultural Intelligence – Relationship Report ===",
                 f"Records analysed : {len(self.df)}"]
        if "region" in self.df.columns:
            lines.append(f"Regions          : {sorted(self.df['region'].dropna().unique().tolist())}")
        if "crop" in self.df.columns:
            lines.append(f"Crops            : {sorted(self.df['crop'].dropna().unique().tolist())}")
        if "avg_temp" in self.df.columns:
            lines.append(f"Temperature range: {self.df['avg_temp'].min():.1f} – {self.df['avg_temp'].max():.1f} °C")
        if "total_rainfall" in self.df.columns:
            lines.append(f"Daily rainfall   : mean {self.df['total_rainfall'].mean():.1f} mm")

        if risk_model is not None and getattr(risk_model, "_is_fitted", False):
            lines.append("\nTop risk drivers:")
            for _, row in self.explain_risk_drivers(risk_model, 5).iterrows():
                lines.append(f"  • {row['feature']:<30} {row['importance']:.3f}")

        if yield_model is not None and getattr(yield_model, "_is_fitted", False):
            lines.append("\nTop yield drivers:")
            for _, row in self.explain_yield_drivers(yield_model, 5).iterrows():
                lines.append(f"  • {row['feature']:<30} {row['importance']:.3f}")

        if anomaly_detector is not None and getattr(anomaly_detector, "_is_fitted", False):
            lines.append("\nMost anomalous observations:")
            for a in self.summarise_anomalies(anomaly_detector, 3):
                lines.append(f"  • {a.get('region')} / {a.get('crop')} @ {a.get('date')} "
                             f"(score={a.get('anomaly_score'):.3f})")

        if "avg_veg_health" in self.df.columns:
            corr = self.top_correlations("avg_veg_health", 4)
            if not corr.empty:
                lines.append("\nStrongest correlates of vegetation health:")
                for _, row in corr.iterrows():
                    lines.append(f"  • {row['feature']:<30} r={row['correlation']:+.3f}")

        lines.append("=" * 55)
        return "\n".join(lines)

    def _require_data(self):
        if self.df is None or self.df.empty:
            raise RuntimeError("Call set_data(df) first.")


In [ ]:
rel = RelationshipModule(df)

print("Region summary:")
display(rel.region_summary())

print("\nCrop summary:")
display(rel.crop_summary())

print("\nCorrelations with vegetation health:")
display(rel.top_correlations("avg_veg_health", top_n=6))

print()
print(rel.narrative_report(
    risk_model=risk,
    yield_model=yield_model,
    anomaly_detector=anomaly,
))


Region summary:


,region,avg_temp,total_rainfall,avg_soil_moisture,avg_pest_threat,avg_veg_health,avg_price
0,Amhara,22.583,5.715,30.440,0.308,0.765,5015.755
1,Oromia,22.195,4.766,32.240,0.312,0.756,5022.592
2,SNNPR,23.639,6.051,31.844,0.321,0.764,5026.881
3,Sidama,22.278,4.513,34.422,0.303,0.761,5018.843
4,Tigray,23.512,4.236,33.622,0.296,0.714,5024.292



Crop summary:


,crop,avg_pest_threat,avg_veg_health,avg_price,price_volatility,total_rainfall
0,Coffee,0.303,0.754,8496.437,67.707,5.044
1,Maize,0.292,0.746,3801.622,64.214,5.144
2,Sorghum,0.317,0.757,3404.405,62.229,5.128
3,Teff,0.312,0.748,5205.223,63.201,4.916
4,Wheat,0.316,0.755,4200.677,64.145,5.049



Correlations with vegetation health:


,feature,correlation
0,production_tons,0.291065
1,total_rainfall,0.242794
2,avg_rainfall,0.240949
3,min_temp,-0.111546
4,max_temp,-0.103978
5,avg_temp,-0.101981



=== Agricultural Intelligence – Relationship Report ===
Records analysed : 1000
Regions          : ['Amhara', 'Oromia', 'SNNPR', 'Sidama', 'Tigray']
Crops            : ['Coffee', 'Maize', 'Sorghum', 'Teff', 'Wheat']
Temperature range: 14.8 – 30.9 °C
Daily rainfall   : mean 5.1 mm

Top risk drivers:
  • avg_pest_threat                0.412
  • avg_veg_health                 0.126
  • avg_temp                       0.062
  • max_temp                       0.061
  • min_temp                       0.060

Top yield drivers:
  • avg_rainfall                   0.435
  • avg_veg_health                 0.375
  • avg_pest_threat                0.119
  • total_rainfall                 0.044
  • avg_humidity                   0.004

Most anomalous observations:
  • Tigray / Coffee @ 2026-07-13 (score=-0.064)
  • Amhara / Teff @ 2026-07-28 (score=-0.054)
  • Oromia / Sorghum @ 2026-07-30 (score=-0.053)

Strongest correlates of vegetation health:
  • production_tons                r=+0.291
  • tota

## 6. Optional — load real HDFS data instead of synthetic

If Kafka producer + HDFS consumer are running and batches exist in `/agri/raw/`,  
uncomment and run the cell below (requires the rest of the backend on `sys.path`).


In [ ]:
# Uncomment when HDFS has data:
#
# import sys
# sys.path.insert(0, ".")   # project root
# from analytics.hdfs_data_loader import get_merged_agri_dataset
#
# df_real = get_merged_agri_dataset()
# if df_real is not None and not df_real.empty:
#     print("Loaded", len(df_real), "rows from HDFS")
#     # Re-fit models on real data:
#     # anomaly.fit(df_real)
#     # risk.fit(df_real)
#     # yield_model.fit(df_real)
# else:
#     print("HDFS empty — keep using synthetic df")

print("Skipped (using synthetic data). Uncomment above when HDFS is ready.")


Skipped (using synthetic data). Uncomment above when HDFS is ready.


## 7. Reload saved models (inference demo)


In [ ]:
import os

# Set and create saved models directory
SAVED_DIR = "models_saved"
os.makedirs(SAVED_DIR, exist_ok=True)

# Fit and save Anomaly Detector
anomaly = AnomalyDetector()
anomaly.fit(df)
anomaly.save(os.path.join(SAVED_DIR, "anomaly_detector.joblib"))

# Fit and save Crop Risk Model
risk = CropRiskModel(n_estimators=150, max_depth=8)
risk.fit(df)
risk.save(os.path.join(SAVED_DIR, "crop_risk_model.joblib"))

# Fit and save Yield Forecaster
yield_model = YieldForecaster(n_estimators=100)
yield_model.fit(df)
yield_model.save(os.path.join(SAVED_DIR, "yield_forecaster.joblib"))

print("\n✅ All 3 models saved successfully!")

[ANOMALY] Saved → models_saved/anomaly_detector.joblib
[CROP-RISK] Hold-out accuracy = 0.925 (train=800, test=200)
[CROP-RISK] Saved → models_saved/crop_risk_model.joblib
[YIELD] R²=0.967, MAE=0.078 t/ha (train=800, test=200)
[YIELD] Saved → models_saved/yield_forecaster.joblib

✅ All 3 models saved successfully!


In [ ]:
anomaly2 = AnomalyDetector.load(os.path.join(SAVED_DIR, "anomaly_detector.joblib"))
risk2 = CropRiskModel.load(os.path.join(SAVED_DIR, "crop_risk_model.joblib"))
yield2 = YieldForecaster.load(os.path.join(SAVED_DIR, "yield_forecaster.joblib"))

sample = df.sample(5, random_state=1).reset_index(drop=True)
print("Inference on 5 random rows:")
out = sample[["region", "crop", "avg_temp", "total_rainfall", "avg_pest_threat"]].copy()
out["anomaly"] = anomaly2.predict(sample)
out["risk"] = risk2.predict(sample)
out["yield_t_ha"] = yield2.predict(sample).round(2)
display(out)


[ANOMALY] Loaded ← models_saved/anomaly_detector.joblib
[CROP-RISK] Loaded ← models_saved/crop_risk_model.joblib
[YIELD] Loaded ← models_saved/yield_forecaster.joblib
Inference on 5 random rows:


,region,crop,avg_temp,total_rainfall,avg_pest_threat,anomaly,risk,yield_t_ha
0,Amhara,Wheat,22.36,2.33,0.472,1,LOW,2.72
1,SNNPR,Coffee,20.46,10.39,0.332,1,LOW,3.51
2,Tigray,Wheat,20.70,10.62,0.122,1,LOW,3.70
3,SNNPR,Coffee,22.96,8.61,0.519,1,LOW,3.18
4,SNNPR,Wheat,22.70,0.00,0.579,1,MEDIUM,2.40


## Viva talking points (Part 4 of the course)

| Decision | Justification |
|----------|---------------|
| Isolation Forest, contamination=0.05 | Unsupervised; ~5 % extreme agro-climatic events expected |
| Risk labels from pest/veg/heat/moisture | Transparent formula tied to signals already in the stream |
| Random Forest | Robust on modest classroom data; free feature importances |
| Yield = production / area | Physically meaningful; falls back only if production missing |
| Batch HDFS reads | Avoids NameNode small-files problem (handled by consumer) |
| Synthetic fallback | Lets you demo offline; same schema as real HDFS features |

**Pipeline shape**

```
Open-Meteo / NASA POWER → producer → Kafka → batch consumer → HDFS
                                                              ↓
                                              this notebook / train_pipeline
                                                              ↓
                                              anomaly · risk · yield models
```
